<a href="https://colab.research.google.com/github/ZeynepBasaran09/veri_bilimi_proje/blob/main/veri_bilimi_proje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------------
# 1. VERİYİ OKUMA VE GENEL BAKIŞ
# ---------------------------------------------------------
print("Veri seti okunuyor...")
df = pd.read_csv('ai4i2020.csv')

# Sınıf dağılımına bakalım
plt.figure(figsize=(6, 4))
sns.countplot(x='Machine failure', data=df, hue='Machine failure', palette='viridis', legend=False)
plt.title('Makine arıza durumu (0: sağlam, 1: arızalı)')
plt.xlabel('Durum')
plt.ylabel('Sayı')
plt.savefig('sinif_dagilimi.png')
plt.close()

# ---------------------------------------------------------
# 2. YENİ ÖZELLİKLER
# ---------------------------------------------------------
df['Temperature_Difference'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power_Approximation'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']
df['Mechanical_Stress'] = df['Tool wear [min]'] * df['Torque [Nm]']

# ---------------------------------------------------------
# 3. VERİYİ HAZIRLAMA
# ---------------------------------------------------------
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])

X = df.drop(['UDI', 'Product ID', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)
y = df['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 4. MODELLER
# ---------------------------------------------------------
models = {
    "Rastgele Orman": RandomForestClassifier(n_estimators=100, random_state=42),
    "Karar Ağacı": DecisionTreeClassifier(random_state=42),
    "K-En Yakın Komşu": KNeighborsClassifier(n_neighbors=5),
    "Lojistik Regresyon": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Yapay Sinir Ağı": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
}

# ---------------------------------------------------------
# 5. EĞİTİM VE SONUÇLAR
# ---------------------------------------------------------
print("\n--- Model sonuçları ---\n")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"[{name}] doğruluk oranı: %{accuracy*100:.2f}")

    print(f"{name} için sınıflandırma raporu:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("-" * 50)

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{name} - Karmaşıklık Matrisi')
    plt.xlabel('Tahmin Edilen')
    plt.ylabel('Gerçek')
    plt.tight_layout()

    dosya_adi = name.replace(" ", "_") + "_matrix.png"
    plt.savefig(dosya_adi)
    plt.close()

print("\nKod çalışmayı tamamladı.")
print("Tüm sonuçlar yukarıda, grafikler PNG olarak kaydedildi.")

Veri seti okunuyor...

--- Model sonuçları ---

[Rastgele Orman] doğruluk oranı: %99.30
Rastgele Orman için sınıflandırma raporu:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1932
           1       0.95      0.84      0.89        68

    accuracy                           0.99      2000
   macro avg       0.97      0.92      0.94      2000
weighted avg       0.99      0.99      0.99      2000

--------------------------------------------------
[Karar Ağacı] doğruluk oranı: %98.50
Karar Ağacı için sınıflandırma raporu:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.78      0.78      0.78        68

    accuracy                           0.98      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.98      0.98      0.98      2000

--------------------------------------------------
[K-En Yakın Komşu] doğruluk oranı: %97